In [1]:
import yfinance as yf
import pandas as pd
from pathlib import Path

/Users/namtran/Documents/stock-pred/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
meta = yf.Ticker("META")
print(meta.info)

{'address1': '1 Meta Way', 'city': 'Menlo Park', 'state': 'CA', 'zip': '94025', 'country': 'United States', 'phone': '650 543 4800', 'website': 'https://www.meta.com', 'industry': 'Internet Content & Information', 'industryKey': 'internet-content-information', 'industryDisp': 'Internet Content & Information', 'sector': 'Communication Services', 'sectorKey': 'communication-services', 'sectorDisp': 'Communication Services', 'longBusinessSummary': "Meta Platforms, Inc. engages in the development of products that enable people to connect and share with friends and family through mobile devices, personal computers, virtual reality (VR) headsets, and AI glasses in the United States, Canada, Europe, Asia-Pacific, and internationally. It operates through two segments, Family of Apps (FoA) and Reality Labs (RL). The FoA segment offers Facebook, which enables people to build community through feed, reels, stories, groups, marketplace, and other; Instagram that brings people closer through Instag

In [3]:
print("Company Sector:", meta.info['sector'])
print("P/E Ratio:", meta.info['trailingPE'])
print("Company Beta:", meta.info['beta'])

Company Sector: Communication Services
P/E Ratio: 20.490723
Company Beta: 1.229


In [4]:
for key, value in meta.info.items():
    print(f"{key}: {value}")

address1: 1 Meta Way
city: Menlo Park
state: CA
zip: 94025
country: United States
phone: 650 543 4800
website: https://www.meta.com
industry: Internet Content & Information
industryKey: internet-content-information
industryDisp: Internet Content & Information
sector: Communication Services
sectorKey: communication-services
sectorDisp: Communication Services
longBusinessSummary: Meta Platforms, Inc. engages in the development of products that enable people to connect and share with friends and family through mobile devices, personal computers, virtual reality (VR) headsets, and AI glasses in the United States, Canada, Europe, Asia-Pacific, and internationally. It operates through two segments, Family of Apps (FoA) and Reality Labs (RL). The FoA segment offers Facebook, which enables people to build community through feed, reels, stories, groups, marketplace, and other; Instagram that brings people closer through Instagram feed, stories, reels, live, and messaging; Messenger, a messaging

In [5]:
data = meta.history(period="max")
print(data.to_string())

                                 Open        High         Low       Close     Volume  Dividends  Stock Splits
Date                                                                                                         
2012-05-18 00:00:00-04:00   41.683948   44.608268   37.669204   37.897202  573576400      0.000           0.0
2012-05-21 00:00:00-04:00   36.212002   36.340871   32.712732   33.733765  168192700      0.000           0.0
2012-05-22 00:00:00-04:00   32.326124   33.297592   30.670662   30.730139  101786600      0.000           0.0
2012-05-23 00:00:00-04:00   31.096919   32.217081   31.087006   31.721434   73600000      0.000           0.0
2012-05-24 00:00:00-04:00   32.663164   32.920899   31.493436   32.742466   50237200      0.000           0.0
2012-05-25 00:00:00-04:00   32.613605   32.663169   30.839186   31.632221   37149800      0.000           0.0
2012-05-29 00:00:00-04:00   31.205963   31.414136   28.400599   28.588945   78063400      0.000           0.0
2012-05-30

In [6]:
#List of stocks that I want to use
TICKERS = ["AAPL", "NVDA"]
#Period
START = "2021-01-01"
END = "2026-06-26"

In [7]:
def fetch_stock(ticker: str) -> pd.DataFrame:
    df = yf.download(ticker, start=START, end=END, auto_adjust=True)
    df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]
    df.index.name = "Date"
    return df[["Open", "High", "Low", "Close", "Volume"]]

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add basic derived columns useful for analysis."""
    df = df.copy()
    df["Daily_Return"]  = df["Close"].pct_change()          # % change day-over-day
    df["MA_20"]         = df["Close"].rolling(20).mean()    # 20-day moving average
    df["MA_50"]         = df["Close"].rolling(50).mean()    # 50-day moving average
    df["Volatility_20"] = df["Daily_Return"].rolling(20).std()  # rolling std of returns
    return df

In [8]:
def main():
    all_data = {}

    repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    raw_dir = repo_root / "data" / "raw"
    raw_dir.mkdir(parents=True, exist_ok=True)

    for ticker in TICKERS:
        raw = fetch_stock(ticker)
        enriched = add_features(raw)
        all_data[ticker] = enriched

        # Save to CSV
        out_path = raw_dir / f"{ticker}_data.csv"
        enriched.to_csv(out_path)
        print(f"  ✓ Saved to {out_path}  ({len(enriched)} rows)")

    # Quick sanity-check preview
    print("\n── AAPL preview ──")
    print(all_data["AAPL"].tail(3).to_string())

    print("\n── NVDA preview ──")
    print(all_data["NVDA"].tail(3).to_string())

    return all_data


if __name__ == "__main__":
    main()

[*********************100%***********************]  1 of 1 completed

  ✓ Saved to /Users/namtran/Documents/stock-pred/data/raw/AAPL_data.csv  (1375 rows)


[*********************100%***********************]  1 of 1 completed

  ✓ Saved to /Users/namtran/Documents/stock-pred/data/raw/NVDA_data.csv  (1375 rows)

── AAPL preview ──
                  Open        High         Low       Close     Volume  Daily_Return       MA_20       MA_50  Volatility_20
Date                                                                                                                      
2026-06-23  297.540009  301.640015  294.179993  294.299988   52010900     -0.009124  302.272502  290.046207       0.015208
2026-06-24  295.359985  299.700012  292.940002  293.079987   53081900     -0.004145  301.510002  290.728578       0.015212
2026-06-25  287.399994  288.799988  273.750000  275.149994  107013700     -0.061178  299.725002  291.059743       0.019863

── NVDA preview ──
                  Open        High         Low       Close     Volume  Daily_Return       MA_20       MA_50  Volatility_20
Date                                                                                                                      
2026-06-23  20